# 02 — Preprocessing & feature engineering

All preprocessing lives in one fitted scikit-learn transformer,
`src.features.CreditFeatureBuilder`, which is embedded in the model pipeline. That
single design choice gives us three things at once: **(1)** no train/serve skew
(the API applies the identical transform), **(2)** no leakage (it's fit on train
only, inside cross-validation), and **(3)** no copy-pasted logic across notebooks,
API, and dashboard.

This notebook documents *what* it does and *why*.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, data
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (8, 4.5)

from src.features import CreditFeatureBuilder, OUTPUT_FEATURES
df = data.load_raw()
X_train, X_val, X_test, y_train, y_val, y_test = data.split_data(df)
print('train/val/test:', len(X_train), len(X_val), len(X_test))

train/val/test: 90000 30000 30000


## Fit on train only, then inspect the learned parameters

Medians and percentile caps are learned from the **training set** so that no
information from validation/test leaks into preprocessing.

In [2]:
fb = CreditFeatureBuilder().fit(X_train, y_train)
print('income median (impute):   ', round(fb.income_median_, 1))
print('dependents median (impute):', fb.dependents_median_)
print('utilization 99% cap:      ', round(fb.util_cap_, 2))
print('debt-ratio 99% cap:       ', round(fb.debt_cap_, 2))

income median (impute):    5400.0
dependents median (impute): 0.0
utilization 99% cap:       1.09
debt-ratio 99% cap:        4989.01


## Before / after on a few messy rows

In [3]:
raw_cols = ['RevolvingUtilizationOfUnsecuredLines','MonthlyIncome','NumberOfDependents','age']
messy = X_train[(X_train['MonthlyIncome'].isna()) | (X_train['RevolvingUtilizationOfUnsecuredLines']>10)].head(3)
print('RAW:')
print(messy[raw_cols].to_string())
print('\nTRANSFORMED (note imputation, capping, new flags & engineered cols):')
print(fb.transform(messy)[['RevolvingUtilizationOfUnsecuredLines','MonthlyIncome','MonthlyIncome_missing',
      'NumberOfDependents','TotalPastDue','IncomePerDependent','MonthlyIncome_log']].round(2).to_string())

RAW:
        RevolvingUtilizationOfUnsecuredLines  MonthlyIncome  NumberOfDependents  age
54799                               0.224978            NaN                 0.0   52
132781                              0.011582            NaN                 0.0   68
22026                               0.407449            NaN                 0.0   59

TRANSFORMED (note imputation, capping, new flags & engineered cols):
        RevolvingUtilizationOfUnsecuredLines  MonthlyIncome  MonthlyIncome_missing  NumberOfDependents  TotalPastDue  IncomePerDependent  MonthlyIncome_log
54799                                   0.22         5400.0                      1                 0.0             0              5400.0               8.59
132781                                  0.01         5400.0                      1                 0.0             0              5400.0               8.59
22026                                   0.41         5400.0                      1                 0.0             0 

## The decisions, and why

| Decision | What | Why |
|---|---|---|
| **Missingness flags** | add `MonthlyIncome_missing`, `NumberOfDependents_missing` *before* imputing | missing income is likely informative (not missing-at-random); the flag lets the model use that signal |
| **Median imputation** | fill income/dependents with the train median | robust to the heavy right tail; simple and leakage-safe |
| **Winsorize ratios** | cap utilization & debt-ratio at the train 99th percentile | a handful of values (50k+) would otherwise dominate splits/scaling |
| **Cap past-due 96/98** | clip the three past-due counts to 20 | 96/98 are a known data glitch, not "96 late payments" |
| **Clip `age`** | floor at 18 | fixes the lone `age=0` row |
| **Engineered features** | `TotalPastDue`, `IncomePerDependent`, `MonthlyIncome_log` | encode interpretable risk concepts directly |

The full output feature set the model sees:

In [4]:
print(len(OUTPUT_FEATURES), 'features:')
for f in OUTPUT_FEATURES: print(' -', f)

15 features:
 - RevolvingUtilizationOfUnsecuredLines
 - age
 - NumberOfTime30-59DaysPastDueNotWorse
 - DebtRatio
 - MonthlyIncome
 - NumberOfOpenCreditLinesAndLoans
 - NumberOfTimes90DaysLate
 - NumberRealEstateLoansOrLines
 - NumberOfTime60-89DaysPastDueNotWorse
 - NumberOfDependents
 - MonthlyIncome_missing
 - NumberOfDependents_missing
 - TotalPastDue
 - IncomePerDependent
 - MonthlyIncome_log


## Handling class imbalance — the choice

Three standard options, and the reasoning behind ours:

- **Oversampling / SMOTE** — synthesises minority rows. Boosts recall at a fixed
  threshold but **distorts predicted probabilities** (the model "thinks" defaults are
  more common than they are) and invents data points in a credit context where that's
  hard to justify.
- **Class weights / `scale_pos_weight`** — reweights the loss. Same calibration
  problem: it inflates predicted probabilities (we verify this empirically in nb 03).
- **Cost-sensitive thresholding on calibrated probabilities** *(our choice for the
  XGBoost model)* — train normally so probabilities stay **calibrated** (they mean
  what they say, which matters for a *risk score*), then move the **decision
  threshold** using an explicit business cost. The linear baseline still uses
  `class_weight='balanced'` for comparison.

We validate this empirically in notebook 03 (the `scale_pos_weight` ablation trades
calibration for marginal recall).

**Next:** `03_modeling` trains and compares the models.